In [ ]:
using LinearAlgebra
using DynamicPolynomials
using SumOfSquares
using MosekTools
using JuMP
using ProgressMeter

using ppt2

include("utils.jl")
using .Utils

In [ ]:
function check_candidate(state, n, m)
    del, v, V = gen_pncp(n, m)
    phi = del * v + 10 * vec(V * V')

    C_phi = poly2mat(phi, n, m)

    mapped = apply_map(state, C_phi, n, m)

    l = minimum(real(eigvals(mapped)))
    return l, del, v, V
end

function witness_search(state, n, m, tries)
    found = false
    l, del, v, V = Inf, nothing, nothing, nothing

    @showprogress for _ in 1:tries
        l, del, v, V = check_candidate(state, n, m)
        if l < -1e-6
            found = true
            break
        end
    end

    if found
        println("Found a witness with minimum eigenvalue: ", l)
        return del, v, V
    end

    println("No witness found after ", tries, " tries.")
    return nothing
end

In [ ]:
# Example1: Entangled
n = 3
m = 3

candidate_state = (1/3) * [
    1 0 0 0 1 0 0 0 1
    0 0 0 0 0 0 0 0 0
    0 0 0 0 0 0 0 0 0
    0 0 0 0 0 0 0 0 0
    1 0 0 0 1 0 0 0 1
    0 0 0 0 0 0 0 0 0
    0 0 0 0 0 0 0 0 0
    0 0 0 0 0 0 0 0 0
    1 0 0 0 1 0 0 0 1
]

witness = witness_search(candidate_state, n, m, 100);

In [ ]:
# Example 2: Entangled
n = 3
m = 4

candidate_state = (1/60) *[
    5  5 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1
    5  5 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1
   -1 -1  5 -1 -1 -1 -1 -1 -1 -1 -1  5
   -1 -1 -1  5 -1 -1 -1 -1 -1  5 -1 -1
   -1 -1 -1 -1  5  5 -1 -1 -1 -1 -1 -1
   -1 -1 -1 -1  5  5 -1 -1 -1 -1 -1 -1
   -1 -1 -1 -1 -1 -1  5 -1  5 -1 -1 -1
   -1 -1 -1 -1 -1 -1 -1  5 -1 -1  5 -1
   -1 -1 -1 -1 -1 -1  5 -1  5 -1 -1 -1
   -1 -1 -1  5 -1 -1 -1 -1 -1  5 -1 -1
   -1 -1 -1 -1 -1 -1 -1  5 -1 -1  5 -1
   -1 -1  5 -1 -1 -1 -1 -1 -1 -1 -1  5
]

witness = witness_search(candidate_state, n, m, 100);

In [ ]:
# Example 3: Separable
n = 3
m = 3

candidate_state = ones(n * m, n * m)

witness = witness_search(candidate_state, n, m, 100);

In [ ]:
# Example 4: Separable
n = 3
m = 3

candidate_state = Utils.rand_sep(n, m)

witness = witness_search(candidate_state, n, m, 100);

In [ ]:
function search(state, n, m, tries)
    del, v, V = nothing, nothing, nothing
    for _ in 1:tries
        l, del, v, V = check_candidate(state, n, m)
        if l < -1e-6
            break
        end
    end
    return del, v, V
end

function test_ppt2(n, m)
    candidate, witness = nothing, nothing
    @showprogress for _ in 1:10
        del, v, V = gen_pncp(n, m)
        phi = del * v + 10 * vec(V * V')

        C_phi = poly2mat(phi, n, m)

        candidate = apply_map(C_phi, C_phi, n, m)

        del2, v2, V2 = search(candidate, n, m, 10)
        if del2 != nothing
            witness = del2 * v2 + 10 * vec(V2 * V2')
            break
        end
    end
    return C_phi, witness
end

In [ ]:
n = 3
m = 3

C_phi, witness = test_ppt2(n, m);

In [ ]:
Utils.mprint(C_phi)

In [ ]:
Utils.mprint(poly2mat(witness, n, m))

In [ ]:
eigvals(apply_map(candidate, poly2mat(witness, n, m), n, m))